# 2 - Training U-Net for Gap-Filling (Generic - Streaming)

**Self-supervised streaming training pipeline** for any Level-3 ocean variable.

This notebook trains a U-Net to fill gaps in satellite ocean observations using **xbatcher streaming** to keep memory bounded. Works for:
- Chlorophyll-a (PACE, Copernicus, CMEMS)
- Sea surface temperature
- Any other gridded ocean variable with cloud/missing data

## Key Features

- **Streaming workflow**: Zarr → xarray/Dask → xbatcher → TensorFlow
- **Memory-bounded**: Never loads full dataset into RAM
- **Spatial chunking**: 40×56 tiles (or configurable)
- **Generic target variable**: Not hardcoded to chlorophyll
- **Self-supervised**: No gap-free truth needed  
- **Synthetic clouds**: Temporally-correlated fake gaps for training/eval

## Workflow

```
Zarr (on-disk)
  ↓ xr.open_zarr with chunks
Lazy xarray/Dask Dataset
  ↓ build_standardized_lazy (no .load())
Lazy standardized channels
  ↓ xbatcher.BatchGenerator
Spatial tiles (time × 40×56)
  ↓ make_tf_gen → tf.data.Dataset
Training batches
  ↓
model.fit()
```

## Setup

In [1]:
# Import the local checkout when running this notebook from the repository.
import sys
from pathlib import Path

for parent in (Path.cwd(), Path.cwd().parent):
    if (parent / "mindthegap").is_dir():
        sys.path.insert(0, str(parent))
        break

dataset = "globcolour"  # pace, globcolour, indian-ocean, or synthetic
region = "arabian sea"  # or [lat_min, lat_max, lon_min, lon_max]
time_slice = None

# Keep validation runs small. Set False for full training. demo_data applies
# the subsetting when smoke_test=True.
SMOKE_TEST = True


### packages

In [ ]:
!pip install -qU "icechunk>=2" "earthaccess>=0.15"

### GPU set up

In [2]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # Reduce TensorFlow verbosity

import numpy as np
import pandas as pd
import xarray as xr
import tensorflow as tf
import matplotlib.pyplot as plt
import mindthegap as mtg

# TensorFlow automatically uses CPU when no GPU is available.
gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

print(f"TensorFlow version: {tf.__version__}")
print(f"Compute device: {'GPU' if gpus else 'CPU'}")
print(f"GPUs available: {len(gpus)}")

2026-08-07 21:48:38.092724: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1786139318.111802   44325 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1786139318.118321   44325 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1786139318.132832   44325 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786139318.132856   44325 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786139318.132858   44325 computation_placer.cc:177] computation placer alr

TensorFlow version: 2.19.1
Compute device: GPU
GPUs available: 1


## 1. Load Your Data

Load your xarray Dataset with **time, lat, lon** dimensions.

### Requirements:
- Target variable (e.g., `chlor_a`, `sst`, `analysed_sst`)
- Cloud/missing flag variable (1 = cloud/missing, 0 = valid data)
- Land flag variable (1 = land, 0 = ocean)
- Optional: Additional predictor variables (SST, winds, salinity, etc.)

### Data Source Examples:

In [3]:
# Load the data. When SMOKE_TEST is set, demo_data returns a small subset.
ds, data_metadata = mtg.demo_data(
    dataset=dataset,
    region=region,
    time_slice=time_slice,
    smoke_test=SMOKE_TEST,
)
print(f"Selected dataset: {data_metadata['dataset']['name']}")
print(f"Dimensions: {dict(ds.sizes)}")


/srv/conda/envs/notebook/lib/python3.12/site-packages/xarray/backends/plugins.py:109: RuntimeWarning: Engine 'argo' loading failed:
cannot import name 'DocumentModifiedShape' from 'botocore.docs.utils' (/srv/conda/envs/notebook/lib/python3.12/site-packages/botocore/docs/utils.py)
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)
/srv/conda/envs/notebook/lib/python3.12/site-packages/zarr/codecs/numcodecs/_codecs.py:141: ZarrUserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)


Selected dataset: GlobColour
Dimensions: {'time': 120, 'lat': 128, 'lon': 128}


## 2. Build Standardized Training data

Use `build_standardized_lazy()` to create standardized predictors **without loading data into memory**.

In [4]:
# Crop to U-Net-compatible dimensions (multiples of 8).
ds = mtg.crop_to_multiple(ds, multiple=8)

print(f"\nAfter cropping to multiple of 8:")
print(f"Dimensions: {dict(ds.sizes)}")



After cropping to multiple of 8:
Dimensions: {'time': 120, 'lat': 128, 'lon': 128}


In [ ]:
# Pipeline configuration
#
# `mtg.Options` is the single, canonical configuration for this run. Everything
# that can be inferred from the loaded dataset (variable names, bounds, tile
# size, batch size) is resolved by `set_data_config(data=ds)`. Downstream cells
# pass the section they own (options.gridder, options.fit, options.split,
# options.data) rather than threading individual arguments through functions.
options = mtg.Options.default()

# The only non-default data choice for chlorophyll: log-transform the target.
options.data.log_target = True   # True for chl, False for SST

# Resolve all data-dependent configuration from the (cropped) dataset and its
# loader metadata in one call. The train/validation split is chosen separately,
# below, so it is not set here.
options.set_data_config(data=ds, metadata=data_metadata)

print(f"Dataset dimensions: {dict(ds.sizes)}")
print(f"Target variable: {options.data.target_variable}")
print(f"Date range: {pd.to_datetime(ds.time.values[0])} to {pd.to_datetime(ds.time.values[-1])}")
print()
print(options)

## Training and validation dates

Select the dates to use for training and validation

In [ ]:
# Choose the training and validation dates. This updates options.split with the
# selected dates so downstream steps do not need floating date variables.
#
# method="random" samples spaced-out dates (min_day_difference apart) for train
# and validation. Use method="manual" with train_slice/val_slice to pick fixed
# windows, e.g.:
#   mtg.train_validation_dates(
#       ds.time, options.split, method="manual",
#       train_slice=slice("1997-01-01", "2000-01-01"),
#       val_slice=slice("2000-01-02", "2001-01-01"),
#   )
mtg.train_validation_dates(ds.time, options.split, method="random", seed=42)

print(f"Split method: {options.split.method}")
print(f"Training dates: {len(options.split.train_dates)}")
print(f"Validation dates: {len(options.split.val_dates)}")
print(f"Training period: {options.split.training_period()}")

## Build lazy standardized dataset

* add n-prev and n-next targets as inputs
* add features (optional) as inputs; standardize
* add flags (land, real cloud, fake cloud)
* rename the target; standardize and log (optional)

In [ ]:
# Variable names, features, log-transform, and lags all come from options.data.
# output_chunks defaults to the gridder tile layout, and the features are
# standardized by default; options.data is populated in place with the resolved
# channel order, standardization, and target mean/std.
ds_std, stats = mtg.build_standardized_lazy(
    ds,
    train_dates=options.split.train_selection(),
    options=options.data,
    gridder=options.gridder,
    add_geo=False,  # Set True to add spherical lat/lon features
)

print(f"\nChannels created ({len(options.data.input_names)} total):")
for i, ch in enumerate(options.data.input_names, 1):
    print(f"  {i}. {ch}")
print(f"\nTarget standardization: mean={options.data.target_mean:.4f}, "
      f"std={options.data.target_std:.4f}")
print(f"\nDataset is LAZY (not in memory): {ds_std.chunks}")
print(f"\nResolved data configuration:")
print(options.data)

## 3. Create xbatcher Streaming Pipeline

Use `mtg.make_xbatcher()` to create tile generators, then wrap with `mtg.make_tf_gen()` for TensorFlow.

In [ ]:
# Build the streaming train/validation TensorFlow datasets in one call.
# make_generator splits ds_std using options.split, tiles with options.gridder,
# and reads the channel order/target from options.data. It returns the datasets
# and the steps-per-epoch so the notebook does not manage ds_train/ds_val.
train_dataset, val_dataset, train_steps, val_steps = mtg.make_generator(
    ds_std,
    options,
)

print("\u2713 Streaming datasets ready")
print(f"  Batch size: {options.fit.batch_size}")
print(f"  Steps per epoch: train={train_steps}, val={val_steps}")

In [ ]:
# TensorFlow dataset creation is handled inside mtg.make_generator above.


## 4. Build U-Net Model

Fully-convolutional U-Net that can accept any spatial size (trains on 40×56, can predict on full domain).

In [ ]:
# Build U-Net (fully-convolutional). The channel count comes from the resolved
# configuration; compilation is handled by mtg.fit_model using options.fit.
num_channels = len(options.data.input_names)
model = mtg.UNet((None, None, num_channels))

model.summary()

tile_lat, tile_lon = options.gridder.tile_size
print(f"\nModel input shape: (batch, {tile_lat}, {tile_lon}, {num_channels})")
print(f"Model output shape: (batch, {tile_lat}, {tile_lon}, 1)")

## 5. Train with Streaming Data

Train using xbatcher streaming—data is loaded one tile at a time, keeping memory bounded.

In [ ]:
# Train using the fit configuration. mtg.fit_model compiles the model with
# options.fit (optimizer, learning rate, loss) and installs an EarlyStopping
# callback using options.fit.patience. The steps come from make_generator.
print("Starting training...")
print(f"  Epochs: {options.fit.epochs}")
print(f"  Batch size: {options.fit.batch_size}")
print(f"  Steps per epoch: train={train_steps}, val={val_steps}")
print(f"  Early stopping patience: {options.fit.patience}")
print("\n" + "="*60)

history = mtg.fit_model(
    model,
    train_dataset,
    options.fit,
    validation_data=val_dataset,
    steps_per_epoch=train_steps,
    validation_steps=val_steps,
    verbose=1,
)

print(f"Best val_loss: {min(history.history['val_loss']):.6f}")
print(f"Final train_loss: {history.history['loss'][-1]:.6f}")

## 6. Visualize Training History

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))

epochs_run = len(history.history['loss'])
ax.plot(history.history['loss'], label='Train Loss')
ax.plot(history.history['val_loss'], label='Val Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title(f'Training History ({epochs_run} epochs)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nBest epoch: {np.argmin(history.history['val_loss']) + 1}")
print(f"Best val_loss: {min(history.history['val_loss']):.6f}")

## 7. Test Prediction (Full Domain)

Load one test frame and predict on the full domain to verify the model works.

In [ ]:
# Pick a test day: the last date in the record that was not used for training
# or validation, falling back to the final available day.
used = set(options.split.train_dates) | set(options.split.val_dates)
available = [str(pd.to_datetime(t).date()) for t in ds_std.time.values]
unused = [d for d in available if d not in used]
test_date = unused[-1] if unused else available[-1]
print(f"Test prediction date: {test_date}")

# Select and load test frame
ds_test = ds_std.sel(time=test_date).load()

# Stack channels in the resolved order.
X_test = np.stack(
    [np.nan_to_num(ds_test[ch].values, nan=0.0) for ch in options.data.input_names],
    axis=-1,
).astype('float32')
X_test = X_test[np.newaxis, ...]  # Add batch dimension

print(f"Test input shape: {X_test.shape}")

# Predict (fully-convolutional model handles any size)
y_pred = model(X_test, training=False).numpy()[0, :, :, 0]

# Unstandardize using the resolved target statistics.
y_pred_orig = y_pred * options.data.target_std + options.data.target_mean

print(f"Prediction shape: {y_pred_orig.shape}")
print(f"Prediction range: [{np.nanmin(y_pred_orig):.4f}, {np.nanmax(y_pred_orig):.4f}]")
print("\n\u2713 Model successfully predicts on full domain")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True, sharey=True)

# Put observations in the same unstandardized units as the prediction.
observed = ds_test['masked_target'].values * options.data.target_std + options.data.target_mean
land = ds_test['land_flag'].values.astype(bool)
observed = np.ma.masked_where(land | ~np.isfinite(observed), observed)
prediction = np.ma.masked_where(land | ~np.isfinite(y_pred_orig), y_pred_orig)

# Use geographic coordinates and one robust scale for both panels.
vmin, vmax = np.nanpercentile(observed.filled(np.nan), [2, 98])
plot_kwargs = dict(cmap='viridis', shading='auto', vmin=vmin, vmax=vmax)
longitude = ds_test['lon'].values
latitude = ds_test['lat'].values

axes[0].set_facecolor('0.75')
axes[0].pcolormesh(longitude, latitude, observed, **plot_kwargs)
axes[0].set_title(f'Observed (with synthetic clouds)\n{test_date}')

axes[1].set_facecolor('0.75')
im = axes[1].pcolormesh(longitude, latitude, prediction, **plot_kwargs)
axes[1].set_title(f'U-Net Prediction\n{test_date}')

for ax in axes:
    ax.set_xlabel('Longitude (degrees east)')
    ax.set_ylabel('Latitude (degrees north)')
    ax.set_aspect('equal')

color_label = 'Log chlorophyll-a' if options.data.log_target else 'Chlorophyll-a'
plt.colorbar(im, ax=axes, label=color_label, fraction=0.02)
fig.subplots_adjust(wspace=0.08, right=0.88)
plt.show()


## 8. Save Model

Save the trained model for later use.

In [ ]:
# Everything the bundle needs is already resolved on options.data, so the
# metadata and the fitting pipeline cannot diverge. Save the complete resolved
# Options with the bundle.
region = {"lat": list(options.data.lat_bounds), "lon": list(options.data.lon_bounds)}
bundle_path = Path("../models") / f"{dataset}-unet-bundle"
metadata_path = mtg.create_model_bundle_metadata(
    bundle_path,
    model_name=f"{options.data.source} U-Net gap filler",
    dataset_name=options.data.source,
    product_id=options.data.product_id,
    region=region,
    training_period=options.split.training_period(),
    input_names=options.data.input_names,
    target_name=options.data.target_name,
    target_units=options.data.target_units,
    expected_input_shape=list(model.input_shape),
    transforms=options.data.transforms,
    standardization=options.data.standardization,
    missing_value_handling=options.data.missing_value_handling,
    limitations=(
        "Validated only for the documented product, region, training period, "
        "channel order, and preprocessing configuration."
    ),
    options=options,
    overwrite=True,
)
print(f"Metadata saved to: {metadata_path}")


### Stop and review the metadata

Open the `model_metadata.yaml` file printed above. Check the dataset, region, training period, input channel order, transforms, and standardization values. **Do not run the next cell until the metadata is correct.**

In [ ]:
mtg.save_model_bundle(
    model,
    bundle_path,
    overwrite=True,
)
print(f"✓ Model bundle saved to: {bundle_path}")

In [ ]:
# Verify that the released artifact loads without rebuilding the U-Net.
loaded_model, loaded_metadata = mtg.load_model_bundle(bundle_path)
round_trip_input = X_test[:1]
expected = model(round_trip_input, training=False).numpy()
actual = loaded_model(round_trip_input, training=False).numpy()
np.testing.assert_allclose(actual, expected, rtol=1e-6, atol=1e-6)
assert [item["name"] for item in loaded_metadata["inputs"]] == options.data.input_names
print("✓ Bundle reload produced equivalent predictions")
